# Children Speech Recognition — Training on Google Colab

Run each cell **in order**. Do not skip any cell.

## Step 1 — Check GPU

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found. Go to Runtime > Change Runtime Type > GPU')

## Step 2 — Mount Google Drive (for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/Children_Speech_Recognition'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
print(f'Drive folder ready: {DRIVE_DIR}')

## Step 3 — Clone Your GitHub Repo

In [ ]:
%cd /content
!git clone https://github.com/Adnan082/Children_Speech_Recognition.git
%cd Children_Speech_Recognition
!ls

## Step 4 — Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 5 — Link Data from Google Drive

In [ ]:
# Your raw data is already on Drive — create a symlink so code can find it
# Change this path to wherever your raw data folder is in your Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/Children_Speech_Recognition/data/raw'

!ln -sfn {DRIVE_DATA_PATH} /content/Children_Speech_Recognition/data/raw

# Verify
!ls data/raw/

## Step 6 — Update Config for Colab Paths

In [ ]:
import yaml

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# point checkpoints and logs to Drive so they survive session resets
config['output']['checkpoint_dir'] = '/content/drive/MyDrive/Children_Speech_Recognition/checkpoints'
config['output']['log_dir'] = '/content/drive/MyDrive/Children_Speech_Recognition/logs'

# cache preprocessed dataframe to Drive — skip re-processing on reconnects
CACHE_PATH = '/content/drive/MyDrive/Children_Speech_Recognition/dataset_cache.csv'
config['data']['cache_path'] = CACHE_PATH

# colab can handle more workers
config['training']['num_workers'] = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)

print('Config updated:')
print(f'  checkpoint_dir: {config["output"]["checkpoint_dir"]}')
print(f'  log_dir:        {config["output"]["log_dir"]}')
print(f'  cache_path:     {CACHE_PATH}')
print(f'  num_workers:    {config["training"]["num_workers"]}')

## Step 7 — Preprocess & Cache Dataset

In [ ]:
import sys
sys.path.insert(0, '/content/Children_Speech_Recognition')

from src.data.preprocess import build_dataset

# First run: processes raw data and saves cache to Drive (~2-3 mins)
# Next runs: loads instantly from cache
df = build_dataset(
    data_dir='data/raw',
    jsonl_path='data/raw/train_word_transcripts.jsonl',
    cache_path=CACHE_PATH
)
print(f'Dataset ready: {len(df):,} samples')

## Step 8 — Verify Model Loads Correctly

In [ ]:
from src.models.model import load_model, load_processor, get_model_info

processor = load_processor('facebook/wav2vec2-base-960h')
model = load_model(
    model_name='facebook/wav2vec2-base-960h',
    vocab_size=processor.tokenizer.vocab_size,
    freeze_feature_encoder=True
)
print()
get_model_info(model)
print('\nAll good — ready to train!')

## Step 9 — Train

In [ ]:
from src.training.train import train
train(config_path='configs/config.yaml')

## Step 10 — Evaluate Best Model

In [ ]:
import os
checkpoint_dir = '/content/drive/MyDrive/Children_Speech_Recognition/checkpoints'
checkpoints = sorted(os.listdir(checkpoint_dir))
print('Available checkpoints:')
for c in checkpoints:
    print(f'  {c}')

In [ ]:
from src.training.evaluate import run_evaluation

best_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])
print(f'Evaluating: {best_checkpoint}')

run_evaluation(
    config_path='configs/config.yaml',
    checkpoint_path=best_checkpoint
)

## Step 11 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = '/content/drive/MyDrive/Children_Speech_Recognition/logs/training_log.csv'
log_df = pd.read_csv(log_path)

plt.figure(figsize=(10, 4))
plt.plot(log_df['epoch'], log_df['train_loss'], label='Train Loss', marker='o')
plt.plot(log_df['epoch'], log_df['val_loss'], label='Val Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('CTC Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Best val loss: {log_df["val_loss"].min():.4f} at epoch {log_df["val_loss"].idxmin()+1}')